### Product Research

In [1]:
from openai import AsyncOpenAI
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool, input_guardrail, GuardrailFunctionOutput, OpenAIChatCompletionsModel
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

In [ ]:

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"""You are a helpful product research assistant. Given the product and the requirements mentioned in the input query, your goal is to come up with a set of web search queries to perform to provide the best 
    product models/brands out in the market. The product can be a electronic item, consumer good, food item, foot wear, apparel, insurance policy, financial instrument, musical instrument, furniture or anything literally. 
    Along with the key requirements highlighted in the input, consider the below points to come up with the web search queries.
1. Pros and cons of the products that you think are best alligned with the input requirements 
2. Customer reviews and reputation
3. Market position
4. Product performance
5. Quality and reliability
6. Prive-value ratio - whether the benefits justify the cost relatve to competitors.
7. customer support

Output {HOW_MANY_SEARCHES} terms to query for."""


class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the input query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [24]:
INSTRUCTIONS = """You are a product research assistant. Your goal is to come up with useful information of top product models/brands based on the input_query. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 
words. Capture the main points. Write succintly, no need to have complete sentences or good 
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the 
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."""

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [25]:
INSTRUCTIONS = (
    """You are a senior researcher tasked with writing a cohesive report for a research query. 
    You will be provided with the original query, and some initial research done by a research assistant.
    You should first come up with an outline for the report that describes the structure and 
    flow of the report. Then, generate a detailed report and return that as your final output. The report should contain a comparison of those top product models/brands by listing their pros and cons in a tabular format.  
    Think of a few metrics to evaluate a winner based on the requirements mentioned in the input query as well as the points mentioned above. 
    The final output should be in markdown format, and it should be lengthy and detailed. Aim for 2-3 pages of content, at least 1000 words."""
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [49]:
@function_tool
def send_email(html_body: str) -> Dict[str, str]:
    """ Send out an email with the given body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tanvi96patil@gmail.com") 
    to_email = To("tanvi96patil@gmail.com") 
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, 'Product Report', content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [50]:
html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [51]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[html_tool, send_email],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name],
)

In [52]:
@function_tool
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

@function_tool
async def perform_searches(input_query: str, search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(input_query, item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(input_query: str, item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

In [53]:
@function_tool
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

    
# async def send_email(report: ReportData):
#     """ Use the email agent to send an email with the report """
#     print("Writing email...")
#     result = await Runner.run(email_agent, report.markdown_report)
#     print("Email sent")
#     return report

In [54]:
tools = [plan_searches, perform_searches, write_report]
handoffs = [email_agent]

In [55]:
product_manager_instructions = f"""
You are a Product Manager. Your goal is to write a research report on highly relevant top brands/models associated with the product and the requirements mentioned in the query. 

Along with the input query requirements, consider the below points to the best report.
    1. Pros and cons of the products that you think are best alligned with the input requirements 
    2. Customer reviews and reputation
    3. Market position
    4. Product performance
    5. Quality and reliability
    6. Prive-value ratio - whether the benefits justify the cost relatve to competitors.
    7. customer support

Follow these steps carefully:
1. Generate {HOW_MANY_SEARCHES} web search queries: Use plan_searches tool to generate these web search queries. Do not proceed until they are ready. You can raise the number of web searches denoted by "HOW_MANY_SEARCHES" by strictly maximum 2 searches only 
if you think you need more information to draft a strong product report. 
 
2. Web search: Use perform_searches tool to extract useful information on the web. 

3. Report writing: Write a report conforming all the requirements in the query and key points mentioned above. 

4. Handoff for Sending: Pass the report to the "Email agent" agent. It will take care of formatting and sending.
"""


product_manager = Agent(
    name="Product Manager",
    instructions=product_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

query ="Protein powder with clean ingredients"

with trace("Automated Product Research"):
    result = await Runner.run(product_manager, query)

Planning searches...
Will perform 1 searches
Searching...
Finished searching
Thinking about report...
Finished writing report


In [10]:
query ="Protein powder with clean ingredients"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(query, search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!
